# **Cifar 10** - Classificação de Imagens no RPi com TFLite
> Adaptado da seção [*Image Classification Fundamentals*](https://mjrovai.github.io/EdgeML_Made_Ease_ebook/raspi/image_classification/image_classification_fund.html) do [Prof. Marcelo Rovai](https://github.com/Mjrovai) no livro [*EdgeML Made Easy*](https://mjrovai.github.io/EdgeML_Made_Ease_ebook/) e do repositório do GitHub [Edge Machine Learning Systems Engineering](https://github.com/Mjrovai/UNIFEI-IESTI05-EDGE_AI/tree/main).

## Introdução

Este notebook é um guia prático para a classificação de imagens utilizando um modelo de aprendizado de máquina no formato TensorFlow Lite (TFLite), otimizado para execução em dispositivos com hardware limitado, como o Raspberry Pi.

O objetivo principal é demonstrar o processo de ponta a ponta, que inclui:

1. Carregamento do Modelo: Inicialmente, carregamos um modelo TFLite pré-treinado e a lista de rótulos (categorias) que ele é capaz de reconhecer.

2. Preparação da Imagem: Em seguida, preparamos uma imagem para a análise, ajustando seu tamanho e normalizando os valores dos pixels para que sejam compatíveis com a entrada esperada pelo modelo.

3. Execução da Inferência: Realizamos a classificação da imagem utilizando o interpretador do TFLite e medimos o tempo necessário para essa tarefa, demonstrando a eficiência do modelo em hardware de baixa potência.

4. Análise dos Resultados: Por fim, interpretamos a saída do modelo para identificar a categoria prevista para a imagem e o nível de confiança dessa previsão.

## Importação de bibliotecas

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tflite_runtime.interpreter as tflite

- A célula abaixo importa o módulo warnings e configura o filtro para ignorar avisos do tipo UserWarning.  
  - Objetivo: reduzir ruído na saída do notebook, evitando que avisos não críticos apareçam durante a execução.  
  - Observação: usar com cautela — ocultar avisos pode mascarar problemas úteis para depuração. Para reativar avisos, remova a linha ou use warnings.filterwarnings('default').

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import tflite_runtime as tflr
print(tflr.__version__)

## Carregamento do modelo TFLite e inspeção de tensores de entrada/saída

Na célula abaixo vamos definir os caminhos dos arquivos necessários para a classificação de imagens:
- `model_path`: Caminho para o modelo TFLite CIFAR treinado na nuvem
- `img_path`: Caminho para a imagem de teste (gato)

In [ ]:
model_path = "./models/cifar10.tflite"
img_path = "./imagens/cat_2.jpg"

In [ ]:
labels = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

Em seguida:
- carregamos o modelo TFLite MobileNet V2,
- inicializamos o interpretador e
- obtemos os detalhes dos tensores de entrada e saída necessários para realizar a inferência.

Esses detalhes ajudam a garantir que:
- a imagem de entrada seja processada corretamente e que
- os resultados da classificação possam ser interpretados.

In [ ]:
# Carregar o modelo TFLite
interpreter = tflite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

# Obter os tensores de entrada e saída
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

In [ ]:
input_details

In [ ]:
output_details

In [ ]:
# Obter o tipo de dados de entrada
input_dtype = input_details[0]['dtype']
input_dtype

- O tipo de dados de entrada é `float32`.
- Isso ocorre porque os pixels foram normalizados antes de serem usados no modelo, tendo um intervalo de [0, 1].
- Portanto, a imagem de entrada também precisa ser normalizada.

## Carregamento e preprocessamento da imagem de teste

- A célula de código abaixo realiza o carregamento de uma imagem a partir do caminho especificado na variável `img_path` utilizando a biblioteca PIL (`Image.open`).
- Assim a imagem pode ser:
    - visualizada,
    - processada e
    - posteriormente utilizada como entrada no modelo de classificação.
    
- O objeto retornado representa a imagem em formato manipulável pelo Python, permitindo operações como redimensionamento, exibição e transformação dos dados para o formato esperado pelo modelo de machine learning.

In [ ]:
# Carregar a imagem
img = Image.open(img_path)

- A célula de código abaixo exibe a imagem carregada anteriormente utilizando a biblioteca Matplotlib.

 Ela cria uma figura com tamanho definido, mostra a imagem na tela e adiciona um título ("Imagem Original") ao gráfico.
 
 - Com isso podemos visualizar a imagem original antes de realizar qualquer pré-processamento ou classificação
    - Nessa etapa você pode conferir se o carregamento foi feito corretamente e se a imagem está adequada para os próximos passos do fluxo de trabalho.

In [ ]:
# Exibir a imagem original
plt.figure(figsize=(8, 8))
plt.imshow(img)
#plt.axis('off')  # Isso desativa os eixos
plt.title("Original Image")
plt.show()

A célula de código abaixo:
- obtém as dimensões da imagem carregada (largura, altura e número de canais de cor), e
- armazena essas informações na variável `shape`.
- Em seguida, exibe o formato da imagem no console

Assim você pode confirmar que a imagem foi carregada corretamente e para verificar se suas dimensões e canais estão adequados para o processamento posterior no modelo de classificação.

In [ ]:
width, height = img.size
channels = len(img.getbands())
shape = (height, width, channels)

print(f"Image shape: {shape}")

Agora realizamos o **pré-processamento** da imagem carregada:
- redimensionando-a para o tamanho esperado pelo modelo (224x224 pixels) e
- adicionando uma dimensão extra para representar o batch.

O resultado é um array pronto para ser utilizado como entrada na rede neural durante a inferência.

In [ ]:
# Preprocessar a imagem
img = img.resize((input_details[0]['shape'][1], input_details[0]['shape'][2]))
input_data = np.expand_dims(img, axis=0)
input_data = (input_data/255.).astype(np.float32)
input_data.shape

A próxima célula exibe o tipo de dado (dtype) do array `input_data`.
- Assim você pode verificar se o formato da entrada coincide com o esperado pelo modelo TFLite (por exemplo `uint8` em modelos quantizados).
    - Se houver discrepância, é necessário converter (`.astype(...)`) antes de passar os dados ao interpretador.

In [ ]:
# Obter o tipo de dados da imagem de entrada
input_data.dtype

O tipo de dados de entrada é `uint8`, que é compatível com o tipo de dados esperado para o modelo.

In [ ]:
new_size = (input_details[0]['shape'][1], input_details[0]['shape'][2])
new_size

In [ ]:
# Exibir a imagem redimensionada
plt.figure(figsize=(3, 3))
plt.imshow(img)
#plt.axis('off')  # Isso desativa os eixos
plt.title("Resized Image "+str(new_size))
plt.show()

## Inferência de modelo

### Inferência

Observe que esse modelo não é quantizado, portanto, os dados de entrada permanecem como `float32`.

In [ ]:
# Obter os parâmetros de quantização da saída
scale, zero_point = output_details[0]['quantization']
scale, zero_point

A próxima célula faz a inferência do modelo TFLite e mede o tempo gasto:
- Inicia um cronômetro, define os dados de entrada no tensor do interpretador e chama `interpreter.invoke()` para executar a inferência.
- Calcula o tempo total de inferência em milissegundos e o imprime formatado.
- OBS: depende de variáveis/objetos previamente definidos (por exemplo, `interpreter`, `input_details` e `input_data`) e fornece a métrica de desempenho usada nas células seguintes.

In [ ]:
# Inferência no RPi
start_time = time.time()
interpreter.set_tensor(input_details[0]['index'], input_data)
interpreter.invoke()
end_time = time.time()
inference_time = (end_time - start_time) * 1000  # Convert to milliseconds
print ("Inference time: {:.1f}ms".format(inference_time))

- Agora recuperamos do interpretador TFLite o tensor de saída gerado pela última inferência.
    - Usamos `output_details[0]['index']` para acessar o índice do tensor de saída e `[0]` para remover a dimensão de batch (ficando apenas o vetor de pontuações por classe).
- OBS: Depende de `interpreter` e `output_details` já inicializados e de `interpreter.invoke()` ter sido executado anteriormente.
- Resultado: um array 1D com as previsões quantizadas (valores brutos) que serão desquantizadas e convertidas em probabilidades nas células seguintes.

In [ ]:
# Obter resultados e mapear para rótulos
predictions = interpreter.get_tensor(output_details[0]['index'])[0]

A célula de código abaixo exibe o vetor de previsões gerado pelo modelo TFLite após a inferência.  
- Esse vetor contém os valores brutos (quantizados) para cada classe possível, representando o grau de correspondência da imagem analisada com cada categoria do modelo.  
- Esses valores ainda não são probabilidades e precisam ser desquantizados e normalizados (softmax) para interpretação final, como será feito nas próximas etapas do notebook.

In [ ]:
predictions

Exibir o formato (shape) do vetor de previsões gerado pelo modelo TFLite.  
- Isso permite verificar quantas classes o modelo está considerando e garante que o processamento posterior (seleção dos melhores resultados, desquantização e aplicação do softmax) será feito sobre o número correto de categorias.  

- Com isso, validamos que a saída da inferência está conforme o esperado antes de prosseguir para a interpretação dos resultados.

In [ ]:
predictions.shape

A célula de código abaixo seleciona os índices dos **top 5 resultados** da classificação, ou seja, as classes que receberam as maiores pontuações do modelo para a imagem analisada.  
- Utiliza a função `np.argsort` para ordenar as previsões em ordem decrescente e extrai os índices correspondentes às classes mais prováveis.  
    - Esses índices serão usados para exibir os nomes das classes e suas probabilidades nas etapas seguintes, facilitando a interpretação dos resultados da inferência.

In [ ]:
# Obter os rótulos dos k melhores resultados
top_k_results = 5
top_k_indices = np.argsort(predictions)[::-1][:top_k_results]
top_k_indices

Vamos inprimir os nomes das classes correspondentes aos índices dos principais resultados da classificação.

In [ ]:
print(labels[3])
print(labels[5])
print(labels[2])
print(labels[6])
print(labels[7])

Na próxima célula imprimiremos os valores quantizados (*raw*) das previsões para as 5 classes mais relevantes identificadas pelo modelo.

Esses valores ainda não foram convertidos para probabilidades (0-1) e precisam passar por desquantização e normalização via softmax para representarem as probabilidades finais de cada classe.


In [ ]:
print (predictions[3])
print (predictions[5])
print (predictions[2])
print (predictions[6])
print (predictions[7])

### Testando com outras imagens 

### Definir uma função geral de Classificação de Imagens

In [ ]:
def image_classification(img_path, model_path, labels, top_k_results=5):
    # load the image
    img = Image.open(img_path)
    plt.figure(figsize=(3, 3))
    plt.imshow(img)
    plt.axis('off')

    # Load the TFLite model
    interpreter = tflite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    
    # Get input and output tensors
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Preprocess
    img = img.resize((input_details[0]['shape'][1], 
                      input_details[0]['shape'][2]))
    input_data = np.expand_dims(img, axis=0)
    input_data = (input_data/255.).astype(np.float32)

    # Inference on Raspi-Zero
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    
    # Obtain results and map them to the classes
    predictions = interpreter.get_tensor(output_details[0]['index'])[0]

    # Get indices of the top k results
    top_index = np.argsort(predictions)[::-1][0]

    print("\n[PREDICTION]        [Prob]\n")
    print("{:20}: {}%".format(
    labels[top_index], (int(predictions[top_index]*100))))

In [ ]:
model_path = "./models/cifar10.tflite"
img_path = "./imagens/cat_2.jpg"
labels = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
image_classification(img_path, model_path, labels, top_k_results=5)

In [ ]:
!ls ./imagens

In [ ]:
img_path = "./imagens/car_1.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/car_2.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/car_3.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/car_4.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/car_5.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/cat_1.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/cat_2.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/dog_1.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/dog_2.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/dog_3.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/ship_1.jpg"
image_classification(img_path, model_path, labels)

In [ ]:
img_path = "./imagens/ship_2.jpg"
image_classification(img_path, model_path, labels)